# N114 · 概率与期望DP

**目标：** 由全概率公式和条件期望建立转移。

**先修：** N098, N081, N131。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 概率质量；期望线性性；吸收状态；边界流失；数值容差；自循环方程。

## 从问题到算法

概率DP传播的是概率质量，离开棋盘的质量不能重新归一化。期望则满足首步方程E[i]=1+∑p(i,j)E[j]，有自循环时不能把它当作DAG直接递归。有限小链可以解线性方程；若有正概率进入永不吸收的闭类，吸收时间期望为无穷大。

## 最小实现

**本章接口：** `knight_probability(n, k, row, col)`、`new21_game(n, k, max_pts)`、`expected_steps_small_chain(transitions)`

In [1]:
from fractions import Fraction
from collections import deque

def knight_probability(n,k,row,col):
    if n<=0 or not (0<=row<n and 0<=col<n): return 0.0
    dp=[[0.0]*n for _ in range(n)]; dp[row][col]=1.0
    moves=[(1,2),(1,-2),(-1,2),(-1,-2),(2,1),(2,-1),(-2,1),(-2,-1)]
    for _ in range(k):
        nxt=[[0.0]*n for _ in range(n)]
        for r in range(n):
            for c in range(n):
                for dr,dc in moves:
                    x,y=r+dr,c+dc
                    if 0<=x<n and 0<=y<n: nxt[x][y]+=dp[r][c]/8
        dp=nxt
    return sum(map(sum,dp))

def new21_game(n,k,max_pts):
    if max_pts<=0 or min(n,k)<0: raise ValueError('invalid game parameters')
    if k==0 or n>=k-1+max_pts: return 1.0
    dp=[0.0]*(n+1); dp[0]=1.0; window=1.0; answer=0.0
    for score in range(1,n+1):
        dp[score]=window/max_pts
        if score<k: window+=dp[score]
        else: answer+=dp[score]
        old=score-max_pts
        if 0<=old<k: window-=dp[old]
    return answer

def expected_steps_small_chain(transitions):
    # Row i contains (next_state, probability); [] is absorbing.
    rows=[[(j,p if isinstance(p,Fraction) else Fraction(str(p))) for j,p in row] for row in transitions]
    n=len(rows); reverse=[[] for _ in rows]
    for i,row in enumerate(rows):
        if row and sum(p for j,p in row)!=1: raise ValueError('nonempty rows must sum exactly to one')
        for j,p in row:
            if not 0<=j<n or p<0: raise ValueError('invalid transition')
            if p>0: reverse[j].append(i)
    def ancestors(seeds):
        seen=set(seeds); q=deque(seen)
        while q:
            for u in reverse[q.popleft()]:
                if u not in seen: seen.add(u); q.append(u)
        return seen
    absorbing={i for i,row in enumerate(rows) if not row}
    can_absorb=ancestors(absorbing)
    infinite=ancestors(set(range(n))-can_absorb)
    finite=[i for i in range(n) if i not in absorbing and i not in infinite]; index={v:i for i,v in enumerate(finite)}
    m=len(finite); a=[[Fraction(0) for _ in range(m+1)] for _ in range(m)]
    for r,u in enumerate(finite):
        a[r][r]=1; a[r][-1]=1
        for v,p in rows[u]:
            if v in index: a[r][index[v]]-=p
    for c in range(m):
        pivot=next(r for r in range(c,m) if a[r][c])
        a[c],a[pivot]=a[pivot],a[c]; divisor=a[c][c]; a[c]=[x/divisor for x in a[c]]
        for r in range(m):
            if r!=c:
                factor=a[r][c]; a[r]=[x-factor*y for x,y in zip(a[r],a[c])]
    answer=[float('inf') if i in infinite else 0.0 for i in range(n)]
    for u,r in index.items(): answer[u]=float(a[r][-1])
    return answer

## 正确性、前提与复杂度

全概率公式按上一步状态分解；棋盘外是失败吸收状态，遗漏的质量正是失败概率。New21只把仍需抽牌的分数放进窗口。有限链先排除可到达非吸收闭类的状态，剩余瞬态子矩阵满足(I−Q)可逆，首步方程有唯一有限解；代码用精确有理数消元后转浮点输出。

**代价：** 骑士O(kn²)时间O(n²)空间；New21 O(n)时间空间；小链图检查O(V+E)，高斯消元O(V³)有理数操作O(V²)空间，位复杂度另计。[]表示吸收；非空行概率要求精确和为1，可传Fraction。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['步数','3×3角落骑士留在棋盘概率'],[(k,knight_probability(3,k,0,0)) for k in range(5)])
chain=[[(0,Fraction(1,2)),(1,Fraction(1,2))],[]]
show_table(['状态','转移','吸收前期望步数'],[(i,row,expected_steps_small_chain(chain)[i]) for i,row in enumerate(chain)])

步数,3×3角落骑士留在棋盘概率
0,1.0
1,0.25
2,0.0625
3,0.015625
4,0.00390625


状态,转移,吸收前期望步数
0,"[(0, Fraction(1, 2)), (1, Fraction(1, 2))]",2.0
1,[],0.0


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from math import isclose,isinf
from functools import cache
assert knight_probability(3,2,0,0)==0.0625
assert knight_probability(3,0,0,0)==1.0
assert expected_steps_small_chain([[(0,Fraction(1,2)),(1,Fraction(1,2))],[]])==[2.0,0.0]
assert expected_steps_small_chain([[(1,1)],[(2,1)],[]])==[2.0,1.0,0.0]
a=expected_steps_small_chain([[(1,0.5),(2,0.5)],[],[(2,1)]])
assert isinf(a[0]) and a[1]==0 and isinf(a[2])
for k in range(7):
    for pts in range(1,6):
        for n in range(12):
            @cache
            def exact(score):
                if score>=k: return Fraction(score<=n)
                return sum((exact(score+x) for x in range(1,pts+1)),Fraction(0))/pts
            got=new21_game(n,k,pts)
            assert -1e-12<=got<=1+1e-12 and isclose(got,float(exact(0)),abs_tol=1e-12)
for k in range(5): assert 0<=knight_probability(4,k,1,1)<=1
print('N114: 所有本章断言通过')

N114: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 区分概率之和和期望之和。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 写出含自循环的期望方程。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 688. Knight Probability in Chessboard（canonical）
- 837. New 21 Game（canonical）
- 808. Soup Servings（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。